In [1]:
import redis
import json
import time

class ChatHistory:
    """基于 Redis 的对话历史管理"""
    
    def __init__(self, host='localhost', port=6379, db=0, prefix='chat'):
        self.r = redis.Redis(host=host, port=port, db=db, decode_responses=True)
        self.prefix = prefix
        # 测试连接
        try:
            self.r.ping()
            print("✅ Redis 连接成功")
        except redis.ConnectionError:
            print("❌ Redis 连接失败，请检查 Redis 服务是否启动")

    def _key(self, session_id):
        return f"{self.prefix}:{session_id}:history"

    def add_message(self, session_id, role, content):
        """追加一条消息到会话历史 (role: user / assistant)"""
        message = {
            "role": role,
            "content": content,
            "timestamp": time.time(),
        }
        # 用 List 存储，天然有序，RPUSH 追加
        self.r.rpush(self._key(session_id), json.dumps(message, ensure_ascii=False))

    def get_history(self, session_id, limit=None):
        """获取会话历史"""
        key = self._key(session_id)
        if limit:
            raw = self.r.lrange(key, -limit, -1)
        else:
            raw = self.r.lrange(key, 0, -1)
        return [json.loads(item) for item in raw]

    def clear_history(self, session_id):
        """清空会话历史"""
        return self.r.delete(self._key(session_id))

    def set_expire(self, session_id, seconds=3600):
        """为会话设置过期时间"""
        self.r.expire(self._key(session_id), seconds)

    def list_sessions(self):
        """列出所有会话 ID"""
        keys = self.r.keys(f"{self.prefix}:*:history")
        sessions = [k.split(":")[1] for k in keys]
        return sessions

    def get_message_count(self, session_id):
        """获取会话消息数量"""
        return self.r.llen(self._key(session_id))


# ---------------- 演示 ----------------
if __name__ == "__main__":
    chat = ChatHistory()

    sid = "user_001"

    # 清空旧数据
    chat.clear_history(sid)

    # 模拟对话
    chat.add_message(sid, "user", "你好，请介绍一下 Redis")
    chat.add_message(sid, "assistant", "Redis 是一个内存数据库，支持多种数据结构...")
    chat.add_message(sid, "user", "它支持哪些数据结构？")
    chat.add_message(sid, "assistant", "支持 String、List、Hash、Set、ZSet 等...")

    # 设置 1 小时过期
    chat.set_expire(sid, 3600)

    # 读取完整历史
    print("\n===== 完整历史 =====")
    for msg in chat.get_history(sid):
        print(f"[{msg['role']}] {msg['content']}")

    # 只读取最近 2 条（适合用于拼接给 LLM 的上下文）
    print("\n===== 最近 2 条 =====")
    for msg in chat.get_history(sid, limit=2):
        print(f"[{msg['role']}] {msg['content']}")

    # 会话信息
    print(f"\n消息数量: {chat.get_message_count(sid)}")
    print(f"所有会话: {chat.list_sessions()}")

✅ Redis 连接成功

===== 完整历史 =====
[user] 你好，请介绍一下 Redis
[assistant] Redis 是一个内存数据库，支持多种数据结构...
[user] 它支持哪些数据结构？
[assistant] 支持 String、List、Hash、Set、ZSet 等...

===== 最近 2 条 =====
[user] 它支持哪些数据结构？
[assistant] 支持 String、List、Hash、Set、ZSet 等...

消息数量: 4
所有会话: ['user_001']
